# Delft3D Physics-Informed Graph Neural Network

This notebook is configured to run on Kaggle. It imports the model from `pignn_model.py` and runs the training loop.

In [ ]:
# 1. Install Dependencies (torch-scatter removed for much faster installation)
!pip install torch-geometric netCDF4 xarray

In [ ]:
import os
import torch
import numpy as np

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 2. Import the PIGNN model
from pignn_model import extract_graph_from_netcdf, RiverPIGNN, physics_loss_swe

### Load Dataset
**Important:** If using Kaggle datasets, replace `netcdf_path` with the actual path. If running directly from git clone, this path is correct.

In [ ]:
# UPDATE THIS PATH IF NEEDED
netcdf_path = "../data/input/FlowFM_net.nc"

try:
    # Load Graph Data and Move to GPU
    node_coords, edge_index, edge_attr, node_z = extract_graph_from_netcdf(netcdf_path)
    node_coords = node_coords.to(device)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    node_z = node_z.to(device)
    
    num_nodes = node_coords.size(0)
    print(f"Graph successfully extracted! Nodes: {num_nodes}, Edges: {edge_index.size(1)}")

except FileNotFoundError:
    print(f"Dataset not found at {netcdf_path}. Please check the path.")
    num_nodes = 0

In [ ]:
if num_nodes > 0:
    # Initialize Model & Optimizer
    model = RiverPIGNN(node_features_dim=5, hidden_dim=64, edge_dim=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    epochs = 10
    dt = 60.0 # Time step in seconds
    
    print("Starting Training Loop...")
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        # Dummy initial state: [eta, u, v, bathymetry, friction]
        # The bathymetry is now correctly loaded from NetNode_z
        dummy_state_t = torch.randn((num_nodes, 5), dtype=torch.float32).to(device)
        dummy_state_t[:, 3] = node_z # Bathymetry from file
        dummy_state_t[:, 0] = node_z + 2.0  # Dummy Water elevation (2m depth)
        dummy_state_t[:, 4] = 0.03 # Friction
        
        # Predict t+dt
        predicted_state_next = model(dummy_state_t, edge_index, edge_attr)
        
        # Compute Physics Loss (SWE residuals)
        loss = physics_loss_swe(dummy_state_t, predicted_state_next, edge_index, edge_attr, dt=dt)
        
        loss.backward()
        optimizer.step()
        
        print(f"Epoch {epoch+1}/{epochs} - Physics Loss: {loss.item():.6f}")
        
    print("Training Complete!")
    
    # Save Model Weights
    torch.save(model.state_dict(), "pignn_weights.pth")
    print("Model weights saved to pignn_weights.pth")